%md
# Eval 02 · Agent evaluation sample
Picks a stratified, reproducible sample of 10 laundering and 10 false-positive alerts for the agent evaluation. Labels stay in aml_eval; only alert IDs leave Databricks.

In [0]:
from pyspark.sql import functions as F

GOLD = "workspace.aml_gold"
SILVER = "workspace.aml_silver"
EVAL = "workspace.aml_eval"

alerts = spark.table(f"{GOLD}.alerts").select("alert_id", "account_id", "priority", "rules_fired")
laundering = spark.table(f"{SILVER}.transactions").filter("is_laundering = 1").select(
    "txn_id", "from_account_id", "to_account_id"
)
patterns = spark.table(f"{EVAL}.pattern_labels").select("txn_id", "pattern_type")

labelled_txns = laundering.join(patterns, on="txn_id", how="left")

account_truth = (
    labelled_txns.select(F.col("from_account_id").alias("account_id"), "pattern_type")
    .union(labelled_txns.select(F.col("to_account_id").alias("account_id"), "pattern_type"))
    .groupBy("account_id")
    .agg(F.collect_set("pattern_type").alias("true_patterns"))
    .withColumn("is_involved", F.lit(True))
)

In [0]:
labelled_alerts = (
    alerts
    .join(account_truth, on="account_id", how="left")
    .fillna(False, subset=["is_involved"])
    .withColumn("true_patterns", F.coalesce("true_patterns", F.array().cast("array<string>")))
    .withColumn("pick_order", F.sha2(F.concat(F.col("alert_id"), F.lit("seed-42")), 256))
)

display(
    labelled_alerts
    .withColumn("has_pattern", F.size("true_patterns") > 0)
    .groupBy("is_involved", "has_pattern")
    .count()
    .orderBy("is_involved", "has_pattern")
)

In [0]:
positives = (
    labelled_alerts
    .filter(F.col("is_involved") & (F.size("true_patterns") > 0))
    .orderBy("pick_order")
    .limit(10)
    .withColumn("sample_group", F.lit("laundering"))
)

negatives = (
    labelled_alerts
    .filter(~F.col("is_involved"))
    .orderBy("pick_order")
    .limit(10)
    .withColumn("sample_group", F.lit("false_positive"))
)

sample = positives.unionByName(negatives).drop("pick_order")

rows = sample.count()
groups = {r["sample_group"]: r["count"] for r in sample.groupBy("sample_group").count().collect()}
unique_ids = sample.select("alert_id").distinct().count()

print(f"Sample size: {rows}")
print(f"Groups:      {groups}")

assert rows == 20, "Expected 20 alerts"
assert groups == {"laundering": 10, "false_positive": 10}, "Expected 10 per group"
assert unique_ids == 20, "Duplicate alerts in sample"

print("Quality gate passed")

(
    sample.write
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{EVAL}.agent_sample")
)

display(sample.select("alert_id", "priority", "sample_group", "true_patterns").orderBy("sample_group"))

In [0]:
ids = [
    r["alert_id"]
    for r in sample.orderBy(F.sha2("alert_id", 256)).select("alert_id").collect()
]
print("\n".join(ids))